In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
PROJECT = "/content/drive/MyDrive/fruit_project"
!mkdir -p "{PROJECT}/raw"
!wget -O "{PROJECT}/raw/apple_disease.zip" "https://data.mendeley.com/public-files/datasets/9zgkwwv9j8/files/66748fa2-402d-4707-9af8-7f7690e065f7/file_downloaded"

In [ ]:
!ls -lh "{PROJECT}/raw"

In [ ]:
!ls -l "{PROJECT}/raw"
!file "{PROJECT}/raw/apple_disease.zip"
!head -c 300 "{PROJECT}/raw/apple_disease.zip"

In [ ]:
!mv "{PROJECT}/raw/apple_disease.zip" "{PROJECT}/raw/apple_disease.rar"
!apt-get -qq install -y unrar

In [ ]:
!mkdir -p /content/raw
!unrar x -idq "{PROJECT}/raw/apple_disease.rar" /content/raw/

In [ ]:
import os
for root, dirs, files in os.walk("/content/raw"):
    depth = root.count(os.sep) - 2
    if depth <= 3:
        print("  " * depth + os.path.basename(root), f"({len(files)} files)")

In [ ]:
import random
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

ROOT = Path("/content/raw/Apple Disease Dataset")
CANDIDATES = ["Healthy", "Black Rot", "Black Pox", "Powdery Mildew", "Anthracnose", "Codling Moth"]
IMG = {".jpg", ".jpeg", ".png"}

fig, axes = plt.subplots(len(CANDIDATES), 5, figsize=(15, 3 * len(CANDIDATES)))
for r, c in enumerate(CANDIDATES):
    files = [f for f in (ROOT / "train" / c).glob("*") if f.suffix.lower() in IMG]
    for k, f in enumerate(random.sample(files, 5)):
        im = Image.open(f).convert("RGB")
        im.thumbnail((400, 400))
        axes[r][k].imshow(im)
        axes[r][k].axis("off")
    axes[r][0].set_title(c, loc="left", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
import re
import numpy as np
from datetime import datetime

pat = re.compile(r"(\d{8})_(\d{6})")

def stamp(path):
    m = pat.search(path.name)
    if not m:
        return None
    return datetime.strptime(m.group(1) + m.group(2), "%Y%m%d%H%M%S").timestamp()

for c in sorted(p.name for p in (ROOT / "train").iterdir() if p.is_dir()):
    train_t = np.array([t for f in (ROOT / "train" / c).glob("*") if (t := stamp(f)) is not None])
    for split in ["validation", "test"]:
        files = list((ROOT / split / c).glob("*"))
        ts = [t for f in files if (t := stamp(f)) is not None]
        if len(train_t) == 0 or not ts:
            print(f"{c:15s} {split:10s} timestamps parsed: {len(ts)}/{len(files)}")
            continue
        gaps = np.array([np.abs(train_t - t).min() for t in ts])
        print(f"{c:15s} {split:10s} parsed {len(ts)}/{len(files)} | "
              f"within 60s of a train photo: {np.mean(gaps <= 60):.0%}")

In [ ]:
import re, random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from PIL import Image

ROOT = Path("/content/raw/Apple Disease Dataset")
KEEP = ["Healthy", "Black Rot", "Black Pox", "Powdery Mildew"]
IMG = {".jpg", ".jpeg", ".png"}
pat = re.compile(r"(\d{8})_(\d{6})")

# 1. What do the filenames WITHOUT a timestamp look like?
for c in KEEP:
    names = [f.name for s in ["train", "validation", "test"]
             for f in (ROOT / s / c).glob("*")
             if f.suffix.lower() in IMG and not pat.search(f.name)]
    print(f"\n{c}: {len(names)} files without a timestamp")
    for n in random.sample(names, min(8, len(names))):
        print("   ", n)

# 2. Validation photos next to their nearest-in-time training photo
def stamp(path):
    m = pat.search(path.name)
    return datetime.strptime(m.group(1) + m.group(2), "%Y%m%d%H%M%S").timestamp() if m else None

def stamped(split, c):
    return [(stamp(f), f) for f in (ROOT / split / c).glob("*")
            if f.suffix.lower() in IMG and stamp(f) is not None]

def show_pairs(c, n=5):
    train, val = stamped("train", c), stamped("validation", c)
    tt = np.array([t for t, _ in train])
    pairs = []
    for t, f in val:
        j = np.abs(tt - t).argmin()
        pairs.append((abs(tt[j] - t), f, train[j][1]))
    gaps = np.array([g for g, _, _ in pairs])
    print(f"\n{c}: median gap to nearest train photo {np.median(gaps):.0f}s | "
          f"within 5s: {np.mean(gaps <= 5):.0%} | within 15s: {np.mean(gaps <= 15):.0%}")
    close = [p for p in pairs if p[0] <= 60]
    fig, axes = plt.subplots(2, n, figsize=(3 * n, 6.4))
    for k, (g, fv, ft) in enumerate(random.sample(close, min(n, len(close)))):
        for row, f in enumerate([fv, ft]):
            im = Image.open(f).convert("RGB"); im.thumbnail((400, 400))
            axes[row][k].imshow(im)
        axes[0][k].set_title(f"{c}: validation, gap {g:.0f}s", fontsize=9)
        axes[1][k].set_title("train, nearest in time", fontsize=9)
    for ax in axes.ravel():
        ax.axis("off")
    plt.tight_layout(); plt.show()

show_pairs("Black Pox")
show_pairs("Healthy")

In [ ]:
import re, random
from pathlib import Path

ROOT = Path("/content/raw/Apple Disease Dataset")
KEEP = ["Healthy", "Black Rot", "Black Pox", "Powdery Mildew"]
IMG = {".jpg", ".jpeg", ".png"}
pat = re.compile(r"(\d{8})_(\d{6})")

for c in KEEP:
    names = [f.name for s in ["train", "validation", "test"]
             for f in (ROOT / s / c).glob("*")
             if f.suffix.lower() in IMG and not pat.search(f.name)]
    print(f"\n{c}: {len(names)} files without a timestamp")
    for n in random.sample(names, min(8, len(names))):
        print("   ", n)

In [ ]:
import re, random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

ROOT = Path("/content/raw/Apple Disease Dataset")
KEEP = ["Healthy", "Black Rot", "Black Pox", "Powdery Mildew"]
IMG = {".jpg", ".jpeg", ".png"}
seq_pat = re.compile(r"^(IMG|DSC)_(\d+)\.(\w+)$")
ts_pat = re.compile(r"(\d{8})_(\d{6})")

def seq_files(split, c):
    out = []
    for f in (ROOT / split / c).glob("*"):
        if f.suffix.lower() in IMG:
            m = seq_pat.match(f.name)
            if m:
                out.append(((m.group(1), m.group(3)), int(m.group(2)), f))   # ((prefix, ext), number, path)
    return out

def nearest_train(c, split):
    by_key = {}
    for key, n, f in seq_files("train", c):
        by_key.setdefault(key, []).append((n, f))
    results = []
    for key, n, f in seq_files(split, c):
        cands = by_key.get(key)
        if cands:
            m, tf = min(cands, key=lambda x: abs(n - x[0]))
            results.append((abs(n - m), f, tf))
    return results

# 1. Files that fit neither naming pattern
for c in KEEP:
    odd = [f.name for s in ["train", "validation", "test"] for f in (ROOT / s / c).glob("*")
           if f.suffix.lower() in IMG and not seq_pat.match(f.name) and not ts_pat.search(f.name)]
    print(f"{c}: {len(odd)} files match neither pattern", odd[:5])

# 2. How close are validation/test photos to a train photo, by sequence number?
print()
for c in KEEP:
    for split in ["validation", "test"]:
        res = nearest_train(c, split)
        total = len([f for f in (ROOT / split / c).glob("*") if f.suffix.lower() in IMG])
        if not res:
            print(f"{c:15s} {split:10s} no numbered photos"); continue
        g = np.array([r[0] for r in res])
        print(f"{c:15s} {split:10s} numbered {len(res)}/{total} | "
              f"train photo within 1: {np.mean(g<=1):.0%} | within 3: {np.mean(g<=3):.0%} | "
              f"within 10: {np.mean(g<=10):.0%}")

# 3. Look at the actual pairs
def show_seq_pairs(c, split="validation", max_gap=2, n=5):
    close = [r for r in nearest_train(c, split) if r[0] <= max_gap]
    print(f"\n{c}: {len(close)} {split} photos have a train photo within {max_gap} numbers")
    if not close:
        return
    pick = random.sample(close, min(n, len(close)))
    fig, axes = plt.subplots(2, len(pick), figsize=(3 * len(pick), 6.4), squeeze=False)
    for k, (g, fv, ft) in enumerate(pick):
        for row, f in enumerate([fv, ft]):
            im = Image.open(f).convert("RGB"); im.thumbnail((400, 400))
            axes[row][k].imshow(im)
        axes[0][k].set_title(f"{c}: {fv.name} (val)", fontsize=8)
        axes[1][k].set_title(f"train: {ft.name}", fontsize=8)
    for ax in axes.ravel():
        ax.axis("off")
    plt.tight_layout(); plt.show()

for c in ["Black Rot", "Powdery Mildew", "Black Pox"]:
    show_seq_pairs(c)

In [ ]:
import re
import numpy as np
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

ROOT = Path("/content/raw/Apple Disease Dataset")
KEEP = ["Healthy", "Black Rot", "Black Pox", "Powdery Mildew"]
IMG = {".jpg", ".jpeg", ".png"}
SPLITS = ["train", "validation", "test"]

seq_pat = re.compile(r"^(IMG|DSC)_(\d+)(?:\s*\(\d+\))?\.(\w+)$")   # tolerates 'IMG_4593 (1).JPG'
ts_pat = re.compile(r"(\d{8})_(\d{6})")

records = []
for split in SPLITS:
    for cls in KEEP:
        for f in (ROOT / split / cls).glob("*"):
            if f.suffix.lower() not in IMG:
                continue
            m, t = seq_pat.match(f.name), ts_pat.search(f.name)
            if m:
                rec = dict(kind="seq", key=(m.group(1), m.group(3)), val=int(m.group(2)))
            elif t:
                rec = dict(kind="ts", key=("ts",),
                           val=datetime.strptime(t.group(1) + t.group(2), "%Y%m%d%H%M%S").timestamp())
            else:
                rec = dict(kind="other", key=("other",), val=0)
            rec.update(path=str(f), cls=cls, orig_split=split)
            records.append(rec)

print(Counter((r["cls"], r["kind"]) for r in records))

def sizes_for(kind, thr):
    buckets = defaultdict(list)
    for r in records:
        if r["kind"] == kind:
            buckets[(r["cls"], r["key"])].append(r["val"])
    out = defaultdict(list)
    for (cls, key), vals in buckets.items():
        vals.sort()
        size = 1
        for a, b in zip(vals, vals[1:]):
            if b - a > thr:
                out[cls].append(size); size = 1
            else:
                size += 1
        out[cls].append(size)
    return out

for kind, thresholds in [("seq", [0, 1, 2, 3, 5]), ("ts", [2, 5, 10, 30, 60])]:
    print(f"\n=== {kind} files ===")
    for thr in thresholds:
        s = sizes_for(kind, thr)
        for cls in KEEP:
            sz = s.get(cls, [])
            if not sz:
                continue
            n = sum(sz)
            big = sum(x for x in sz if x > 10) / n
            print(f"gap<={thr:<3} {cls:15s} images {n:5d} groups {len(sz):5d} "
                  f"median {int(np.median(sz)):2d} max {max(sz):4d} in groups>10: {big:.0%}")

In [ ]:
import random
from collections import defaultdict, Counter

LINK = {"seq": 2, "ts": 5}                  # max number-gap / seconds-gap that still counts as "same apple"
TARGET = {"train": 0.70, "val": 0.15, "test": 0.15}
SEED = 42

# 1. Chain photos into groups (a photo joins a group if it is close to the previous one)
buckets = defaultdict(list)
for r in records:
    buckets[(r["cls"], r["kind"], r["key"])].append(r)

groups = []
for (cls, kind, key), recs in buckets.items():
    recs = sorted(recs, key=lambda r: r["val"])
    thr = LINK.get(kind, -1)                # unknown filename types: every photo is its own group
    cur = [recs[0]]
    for prev, r in zip(recs, recs[1:]):
        if r["val"] - prev["val"] <= thr:
            cur.append(r)
        else:
            groups.append(cur); cur = [r]
    groups.append(cur)

# 2. Assign WHOLE groups to splits, class by class, biggest groups first
rng = random.Random(SEED)
assignment = {}                             # file path -> "train" / "val" / "test"
for cls in KEEP:
    cg = [g for g in groups if g[0]["cls"] == cls]
    rng.shuffle(cg)
    cg.sort(key=len, reverse=True)          # stable sort: ties stay in random order
    n_total = sum(len(g) for g in cg)
    target = {s: TARGET[s] * n_total for s in TARGET}
    filled = {s: 0 for s in TARGET}
    for g in cg:
        s = max(TARGET, key=lambda s: target[s] - filled[s])    # the split furthest below its target
        filled[s] += len(g)
        for r in g:
            assignment[r["path"]] = s

# 3. Checks
for g in groups:
    assert len({assignment[r["path"]] for r in g}) == 1
print("OK: no group is split across train/val/test\n")

counts = Counter((r["cls"], assignment[r["path"]]) for r in records)
for cls in KEEP:
    n = sum(counts[(cls, s)] for s in TARGET)
    print(f"{cls:15s} " + " | ".join(f"{s} {counts[(cls, s)]:4d} ({counts[(cls, s)]/n:.0%})" for s in TARGET))
print("\ngroups per class:", {cls: sum(1 for g in groups if g[0]['cls'] == cls) for cls in KEEP})

In [ ]:
import os, json, shutil
from pathlib import Path
from torchvision import datasets

DATA = "/content/dataset"
PROJECT = "/content/drive/MyDrive/fruit_project"

# make sure no two files would overwrite each other
names = Counter((r["cls"], Path(r["path"]).name) for r in records)
print("filename collisions:", [k for k, v in names.items() if v > 1])    # expect []

if os.path.exists(DATA):
    shutil.rmtree(DATA)
for r in records:
    dst = Path(DATA) / r["cls"] / Path(r["path"]).name
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(r["path"], dst)

probe = datasets.ImageFolder(DATA)
print(probe.class_to_idx)
rel_to_idx = {Path(p).relative_to(DATA).as_posix(): i for i, (p, _) in enumerate(probe.samples)}

split = {"train": [], "val": [], "test": []}
for r in records:
    split[assignment[r["path"]]].append(rel_to_idx[f"{r['cls']}/{Path(r['path']).name}"])
split["class_to_idx"] = probe.class_to_idx

with open(f"{PROJECT}/split.json", "w") as f:
    json.dump(split, f)
shutil.make_archive(f"{PROJECT}/dataset", "zip", DATA)
print({k: len(v) for k, v in split.items() if k != "class_to_idx"})

In [ ]:
import numpy as np
import torch, torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import DataLoader
from torchvision import models, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
tfm = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

ds = datasets.ImageFolder(DATA, transform=tfm)
net = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
net.fc = nn.Identity()                      # output the 512-number image "fingerprint"
net = net.to(device).eval()

feats = []
with torch.no_grad():
    for x, _ in DataLoader(ds, batch_size=64, num_workers=2):
        feats.append(nn.functional.normalize(net(x.to(device)), dim=1).cpu())
feats = torch.cat(feats)

def nearest_train(sp):
    tr = feats[sp["train"]]
    return {name: (feats[sp[name]] @ tr.T).max(dim=1) for name in ["val", "test"]}

authors = {"train": [], "val": [], "test": []}
name_map = {"train": "train", "validation": "val", "test": "test"}
for r in records:
    authors[name_map[r["orig_split"]]].append(rel_to_idx[f"{r['cls']}/{Path(r['path']).name}"])

for label, sp in [("authors' split", authors), ("our grouped split", split)]:
    res = nearest_train(sp)
    for name in ["val", "test"]:
        s = res[name].values.numpy()
        print(f"{label:18s} {name:5s} median {np.median(s):.3f} | 90th pct {np.percentile(s, 90):.3f} | above 0.95: {np.mean(s > 0.95):.0%}")

# the 5 validation photos most similar to a training photo, in OUR split
res = nearest_train(split)["val"]
order = np.argsort(-res.values.numpy())[:5]
fig, axes = plt.subplots(2, 5, figsize=(15, 6.4))
for k, o in enumerate(order):
    for row, idx in enumerate([split["val"][o], split["train"][res.indices[o].item()]]):
        im = Image.open(ds.samples[idx][0]).convert("RGB"); im.thumbnail((400, 400))
        axes[row][k].imshow(im); axes[row][k].axis("off")
    axes[0][k].set_title(f"val, similarity {res.values[o]:.2f}", fontsize=9)
    axes[1][k].set_title("nearest train photo", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
import numpy as np
from pathlib import Path
from collections import defaultdict

path_to_idx = {r["path"]: rel_to_idx[f"{r['cls']}/{Path(r['path']).name}"] for r in records}
rec_by_idx = {path_to_idx[r["path"]]: r for r in records}

def merged_groups(T):
    """Filename-based groups, plus a link between any two same-class photos with similarity >= T."""
    parent = list(range(len(ds)))
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a
    def union(a, b):
        parent[find(a)] = find(b)

    for g in groups:                                   # links from the filename rules
        idxs = [path_to_idx[r["path"]] for r in g]
        for i in idxs[1:]:
            union(idxs[0], i)

    for cls in KEEP:                                   # links from image similarity
        idxs = np.array([i for i, r in rec_by_idx.items() if r["cls"] == cls])
        f = feats[idxs]
        sim = (f @ f.T).numpy()
        np.fill_diagonal(sim, 0)
        a_s, b_s = np.where(np.triu(sim) >= T)
        for a, b in zip(a_s, b_s):
            union(idxs[a], idxs[b])

    comp = defaultdict(list)
    for i in rec_by_idx:
        comp[find(i)].append(i)
    return list(comp.values())

for T in [1.01, 0.97, 0.96, 0.95, 0.94, 0.93]:         # 1.01 = no similarity links (filename rules only)
    gs = merged_groups(T)
    print(f"\nthreshold {T}")
    for cls in KEEP:
        sz = [len(g) for g in gs if rec_by_idx[g[0]]["cls"] == cls]
        n = sum(sz)
        print(f"  {cls:15s} groups {len(sz):4d} | largest {max(sz):4d} ({max(sz)/n:.0%} of class) "
              f"| images in groups >20: {sum(x for x in sz if x > 20)/n:.0%}")

In [ ]:
import json, random
from collections import Counter

T = 0.96                                              # <- your choice from Cell 6
gs = merged_groups(T)

print("T =", T)
for cls in KEEP:
    sz = [len(g) for g in gs if rec_by_idx[g[0]]["cls"] == cls]
    print(f"  {cls:15s} groups {len(sz)} | largest {max(sz)}")

rng = random.Random(42)
assign = {}                                            # image index -> "train" / "val" / "test"
for cls in KEEP:
    cg = [g for g in gs if rec_by_idx[g[0]]["cls"] == cls]
    rng.shuffle(cg)
    cg.sort(key=len, reverse=True)
    total = sum(len(g) for g in cg)
    target = {"train": 0.70 * total, "val": 0.15 * total, "test": 0.15 * total}
    filled = {s: 0 for s in target}
    for g in cg:
        s = max(target, key=lambda s: target[s] - filled[s])
        filled[s] += len(g)
        for i in g:
            assign[i] = s

for g in gs:
    assert len({assign[i] for i in g}) == 1
print("OK: no group is split across train/val/test\n")

counts = Counter((rec_by_idx[i]["cls"], s) for i, s in assign.items())
for cls in KEEP:
    n = sum(counts[(cls, s)] for s in ["train", "val", "test"])
    print(f"{cls:15s} " + " | ".join(f"{s} {counts[(cls, s)]:4d} ({counts[(cls, s)]/n:.0%})" for s in ["train", "val", "test"]))

split2 = {s: sorted(i for i, v in assign.items() if v == s) for s in ["train", "val", "test"]}
split2["class_to_idx"] = probe.class_to_idx
with open(f"{PROJECT}/split.json", "w") as f:
    json.dump(split2, f)
print("\nsaved new split.json")

# The closest validation/train pairs that remain. After merging, none can be above T.
tr = feats[split2["train"]]
sims = feats[split2["val"]] @ tr.T
best = sims.max(dim=1)
order = np.argsort(-best.values.numpy())[:5]
fig, axes = plt.subplots(2, 5, figsize=(15, 6.4))
for k, o in enumerate(order):
    for row, idx in enumerate([split2["val"][o], split2["train"][best.indices[o].item()]]):
        im = Image.open(ds.samples[idx][0]).convert("RGB"); im.thumbnail((400, 400))
        axes[row][k].imshow(im); axes[row][k].axis("off")
    axes[0][k].set_title(f"val, similarity {best.values[o]:.2f}", fontsize=9)
    axes[1][k].set_title("nearest train photo", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

tr = feats[split2["train"]]

# 1. How many val/test photos still have a close train neighbour?
print("share of photos whose nearest train photo is above each similarity:")
for name in ["val", "test"]:
    best = (feats[split2[name]] @ tr.T).max(dim=1).values.numpy()
    print(f"  {name:5s}", {t: f"{np.mean(best >= t):.1%}" for t in [0.95, 0.94, 0.93, 0.92]})

# 2. Pairs from further down the ranking (the 5 you saw were ranks 1-5)
sims = feats[split2["val"]] @ tr.T
best = sims.max(dim=1)
order = np.argsort(-best.values.numpy())

def show_rank(start, n=5):
    fig, axes = plt.subplots(2, n, figsize=(3 * n, 6.4))
    for k, o in enumerate(order[start:start + n]):
        for row, idx in enumerate([split2["val"][o], split2["train"][best.indices[o].item()]]):
            im = Image.open(ds.samples[idx][0]).convert("RGB"); im.thumbnail((400, 400))
            axes[row][k].imshow(im); axes[row][k].axis("off")
        axes[0][k].set_title(f"val rank {start+k+1}, sim {best.values[o]:.3f}", fontsize=9)
        axes[1][k].set_title("nearest train photo", fontsize=9)
    plt.tight_layout(); plt.show()

show_rank(20)    # roughly the top 5% most similar
show_rank(50)    # roughly the top 10%


In [ ]:
import json, shutil, os

PROJECT = "/content/drive/MyDrive/fruit_project"
shutil.copy(f"{PROJECT}/split.json", f"{PROJECT}/split_FROZEN.json")

with open(f"{PROJECT}/split.json") as f:
    s = json.load(f)
print("class_to_idx:", s["class_to_idx"])
print({k: len(v) for k, v in s.items() if k != "class_to_idx"})
print("dataset.zip on Drive:", os.path.exists(f"{PROJECT}/dataset.zip"),
      f"{os.path.getsize(f'{PROJECT}/dataset.zip')/1e9:.2f} GB")

In [ ]:
import csv, json
from pathlib import Path

PROJECT = "/content/drive/MyDrive/fruit_project"
s = json.load(open(f"{PROJECT}/split.json"))

# 1. Confirm this is the T=0.96 grouped split: no merged group may span two splits
where = {i: name for name in ["train", "val", "test"] for i in s[name]}
gs = merged_groups(0.96)
bad = sum(1 for g in gs if len({where[i] for i in g}) > 1)
print("groups spanning more than one split:", bad)        # must be 0

# 2. Save a human-readable manifest: one row per image
with open(f"{PROJECT}/split_manifest.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["file", "class", "split"])
    for name in ["train", "val", "test"]:
        for i in s[name]:
            path, label = ds.samples[i]
            w.writerow([Path(path).relative_to("/content/dataset").as_posix(), ds.classes[label], name])
print("manifest rows:", sum(len(s[k]) for k in ["train", "val", "test"]))